# Análise de Churn de Clientes Bancários

Este notebook realiza uma análise exploratória de dados de clientes bancários com foco em entender padrões relacionados ao churn.

## Objetivo

Analisar o perfil dos clientes do banco, comparar clientes que cancelaram e não cancelaram o serviço, e identificar características associadas ao churn.

## Perguntas de Negócio

1. Qual é o perfil geral dos clientes do banco?
2. Quais diferenças aparecem entre clientes que cancelaram e clientes que permaneceram?
3. O comportamento dos clientes muda entre Alemanha, França e Espanha?
4. Quais grupos de clientes parecem ter maior risco de churn?

In [155]:
import pandas as pd
import plotly.express as px

In [156]:
# Carregando a base tratada

df = pd.read_csv("../data/bank_churn_clear.csv")
df.head()

,customerid,creditscore,geography,gender,age,estimatedsalary,balance,numofproducts,hascrcard,isactivemember,exited,tenure,churn
0,15634602,619,France,Female,42.0,101348.88,0.00,1,1,1,1,2,1
1,15647311,608,Spain,Female,41.0,112542.58,83807.86,1,1,1,0,1,0
2,15619304,502,France,Female,42.0,113931.57,159660.80,3,0,0,1,8,1
3,15701354,699,France,Female,39.0,93826.63,0.00,2,0,0,0,1,0
4,15737888,850,Spain,Female,43.0,79084.10,125510.82,1,1,1,0,2,0


In [157]:
# Conferindo a base tratada

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   customerid       10000 non-null  int64  
 1   creditscore      10000 non-null  int64  
 2   geography        10000 non-null  object 
 3   gender           10000 non-null  object 
 4   age              10000 non-null  float64
 5   estimatedsalary  10000 non-null  float64
 6   balance          10000 non-null  float64
 7   numofproducts    10000 non-null  int64  
 8   hascrcard        10000 non-null  int64  
 9   isactivemember   10000 non-null  int64  
 10  exited           10000 non-null  int64  
 11  tenure           10000 non-null  int64  
 12  churn            10000 non-null  int64  
dtypes: float64(3), int64(8), object(2)
memory usage: 1015.8+ KB


## 1. Qual é o perfil geral dos clientes do banco?

In [158]:
# Aqui vamos entender as estatísticas gerais das váriaveis númericas da base
df[["age", "creditscore", "estimatedsalary", "balance", "numofproducts", "tenure"]].describe().round(2)

,age,creditscore,estimatedsalary,balance,numofproducts,tenure
count,10000.00,10000.00,10000.00,10000.00,10000.00,10000.00
mean,38.92,650.53,100092.27,76485.89,1.53,5.01
std,10.49,96.65,57510.15,62397.41,0.58,2.89
min,18.00,350.00,11.58,0.00,1.00,0.00
25%,32.00,584.00,51002.11,0.00,1.00,3.00
50%,37.00,652.00,100236.02,97198.54,1.00,5.00
75%,44.00,718.00,149388.25,127644.24,2.00,7.00
max,92.00,850.00,199992.48,250898.09,4.00,10.00


In [159]:
# Analisando a concentração da base por país.
print(df["geography"].value_counts())
print(df["geography"].value_counts(normalize=True))

geography
France     5014
Germany    2509
Spain      2477
Name: count, dtype: int64
geography
France     0.5014
Germany    0.2509
Spain      0.2477
Name: proportion, dtype: float64


In [160]:
# Analisando a quantidade de clientes por gênero

print(df["gender"].value_counts())
print(df["gender"].value_counts(normalize=True))

gender
Male      5457
Female    4543
Name: count, dtype: int64
gender
Male      0.5457
Female    0.4543
Name: proportion, dtype: float64


In [161]:
# Criando um grafico para entender a dispersão por idade dos clientes

grafico = px.histogram(df, x="age")
grafico.show()

### Insight — Perfil geral dos clientes

A base analisada possui 10.000 clientes. 
A França concentra a maior parte da base, com 5.014 clientes, representando 50,14% do total. Alemanha e Espanha possuem participações semelhantes, com aproximadamente 25% cada.

Em relação ao gênero, a base possui uma leve maioria de clientes homens, que representam 54,57% dos registros.

A idade média dos clientes é de aproximadamente 39 anos, e metade dos clientes está entre 32 e 44 anos. Sendo assim podemos notar que a base se concentra em adultos de meia idade, sem predominância extrema de clientes muito jovens ou muito idosos.

## 2. Quais diferenças aparecem entre clientes que cancelaram e clientes que permaneceram?


In [162]:
print(df["churn"].value_counts())
print(df["churn"].value_counts(normalize=True))

churn
0    7963
1    2037
Name: count, dtype: int64
churn
0    0.7963
1    0.2037
Name: proportion, dtype: float64


In [163]:
taxa_churn = df["churn"].mean() * 100
print(f"A taxa geral de churn da tabela é de {taxa_churn.round(2)}")

A taxa geral de churn da tabela é de 20.37


In [164]:
# Aqui vopu comparar as médias entre clientes que ficaram e que cancelaram

df.groupby("churn")[["age", "creditscore", "estimatedsalary", "balance", "numofproducts", "tenure"]].mean().round(2)

,age,creditscore,estimatedsalary,balance,numofproducts,tenure
churn,,,,,,
0,37.41,651.85,99740.94,72745.30,1.54,5.03
1,44.84,645.35,101465.68,91108.54,1.48,4.93


In [165]:
# Agora vou comparar entre os clientes ativos e inativos qual a maior taxa de cancelamento
print(df.groupby("isactivemember")["churn"].mean().round(2))

# Entre os clientes inativos (0), 27% cancelaram.
# Entre os clientes ativos (1), 14% cancelaram.

isactivemember
0    0.27
1    0.14
Name: churn, dtype: float64


In [166]:
# Calculando a taxa de churn por quantidade de produtos

print(df.groupby("numofproducts")["churn"].mean().round(2))
df.groupby("numofproducts")["churn"].agg(["count", "sum", "mean"])

numofproducts
1    0.28
2    0.08
3    0.83
4    1.00
Name: churn, dtype: float64


,count,sum,mean
numofproducts,,,
1,5084,1409,0.277144
2,4590,348,0.075817
3,266,220,0.827068
4,60,60,1.000000


In [167]:
# Criando um grafico para entender a dispersão de churn por idade dos clientes

graf_idade_churn = px.histogram(df, x="age", color="churn", color_discrete_map={0:"blue",1:"red"})
graf_idade_churn.show()

In [168]:
# Criando um grafico para entender a dispersão de churn por saldo dos clientes

graf_saldo_churn = px.histogram(df, x="balance", color="churn", color_discrete_map={0:"blue",1:"red"})
graf_saldo_churn.show()

### Insight — Diferenças entre clientes que cancelaram e permaneceram

A taxa geral de churn da base é de 20,37%, o que representa 2.037 clientes cancelados em um total de 10.000 clientes.

Clientes inativos possuem uma taxa de cancelamento maior que clientes ativos. Entre os clientes inativos, 27% cancelaram, enquanto entre os clientes ativos esse valor foi de 14%. Essa diferença de 13 pontos percentuais indica que clientes com menor interação com o banco podem ter maior chance de cancelamento.

Na análise da quantidade de produtos, clientes com 3 e 4 produtos apresentaram as maiores taxas de cancelamento. Porém, esses grupos possuem poucos clientes na base. Já clientes com apenas 1 produto tiveram o maior número absoluto de cancelamentos, com 1.409 clientes. Sendo assim, clientes com muitos produtos chamam atenção pela taxa de churn, enquanto clientes com apenas um produto representam o maior impacto em quantidade.

## 3. O comportamento dos clientes muda entre Alemanha, França e Espanha?


In [169]:
# Calculando a taxa de churn por país

print(df.groupby("geography")["churn"].mean().round(2))
df.groupby("geography")["churn"].agg(["count", "sum", "mean"])

geography
France     0.16
Germany    0.32
Spain      0.17
Name: churn, dtype: float64


,count,sum,mean
geography,,,
France,5014,810,0.161548
Germany,2509,814,0.324432
Spain,2477,413,0.166734


In [170]:
# Irei calcular médias especificas comparando entre os países para entender como é o comportamento de cada um
# Saldo médio por país
print("Saldo médio por país")
print(df.groupby("geography")["balance"].mean().round(2))

# Média de clientes mais ativos por país
print("\nMédia de clientes mais ativos por país")
print(df.groupby("geography")["isactivemember"].mean().round(2))

# Média de produtos por país
print("\nMédia de produtos por país")
print(df.groupby("geography")["numofproducts"].mean().round(2))

Saldo médio por país
geography
France      62092.64
Germany    119730.12
Spain       61818.15
Name: balance, dtype: float64

Média de clientes mais ativos por país
geography
France     0.52
Germany    0.50
Spain      0.53
Name: isactivemember, dtype: float64

Média de produtos por país
geography
France     1.53
Germany    1.52
Spain      1.54
Name: numofproducts, dtype: float64


In [171]:
# Gráfico com a distribuição de saldo por país
graf_saldo_pais = px.histogram(df, x="balance", color="geography")
graf_saldo_pais.show()

### Insight — Diferenças por país

A Alemanha apresenta a maior taxa de churn entre os países analisados.

Os clientes da Alemanha possuem o maior saldo médio entre os países analisados.

França e Espanha concentram mais clientes com saldo arelativamente mais baixo.

## 4. Quais grupos de clientes parecem ter maior risco de churn?

In [172]:
# Irei separar por grupos para identificar quais tem maior taxa de cancelamento calculando a média entre eles

# Faixa etaria

df["faixa_etaria"] = pd.cut(df["age"], bins=[0,30,40,50,60,100], labels=["Até 30", "31-40", "41-50", "51-60", "Acima de 60"])

df.groupby("faixa_etaria", observed=False)["churn"].mean().round(2)

faixa_etaria
Até 30         0.08
31-40          0.12
41-50          0.34
51-60          0.56
Acima de 60    0.25
Name: churn, dtype: float64

In [173]:
# Clientes Ativos

df.groupby("isactivemember")["churn"].mean().round(2)

isactivemember
0    0.27
1    0.14
Name: churn, dtype: float64

In [174]:
# Quantidade de produtos

df.groupby("numofproducts")["churn"].agg(["count", "sum", "mean"]).round(2)

,count,sum,mean
numofproducts,,,
1,5084,1409,0.28
2,4590,348,0.08
3,266,220,0.83
4,60,60,1.00


### Insight — Grupos com maior risco de churn

Com base nas análises feitas até aqui, foi possivel identificar alguns grupos aparecem mais associados ao churn:

- Clientes inativos apresentaram uma taxa de cancelamento maior do que clientes ativos.
- Clientes da Alemanha tiveram a maior taxa de churn entre os países analisados.
- Clientes com 3 ou 4 produtos apresentaram taxas de cancelamento mais altas, porém são grupos menores dentro da base.
- Clientes com apenas 1 produto tiveram o maior número de cancelamentos, por serem um grupo maior de clientes.
- Clientes com idade média mais alta demonstram ter uma taxa de churn maior.